In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler


In [ ]:
CSV_PATH = "./raw_loan_dataset.csv"


In [ ]:
# 1) Load + initial snapshot
# --------------------------------
df = pd.read_csv(CSV_PATH)

print("\n=== INITIAL HEAD ===")
print(df.head())

print("\n=== INITIAL INFO ===")
print(df.info())

print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())


=== INITIAL HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  

=== INITIAL INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount 

In [ ]:
# 2) Clean currency formatting
# --------------------------------
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

In [ ]:
# 3) Normalize categorical typos BEFORE imputation
# --------------------------------
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}
df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

print(df["HasCollateral"].value_counts())
print(df["PreviousDefaults"].value_counts())
print(df["Approved"].value_counts())

HasCollateral
No     51
Yes    50
Name: count, dtype: int64
PreviousDefaults
No     86
Yes    15
Name: count, dtype: int64
Approved
Yes    68
No     35
Name: count, dtype: int64


In [ ]:
# 4) Impute missing values
# --------------------------------
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])


print(df.isnull().sum())

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            103 non-null    float64
 1   CreditScore       103 non-null    float64
 2   EmploymentYears   103 non-null    float64
 3   LoanAmount        103 non-null    float64
 4   HasCollateral     103 non-null    object 
 5   PreviousDefaults  103 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB
None


In [ ]:
# 5) Remove duplicates
# --------------------------------
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


In [ ]:
# 6) IQR capping on numeric columns
# L3: clip extreme values to the IQR fence — same idea as house data-processing.py
# --------------------------------
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

print(df.head(10))

     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0             1.10     25800.0           Yes   
1   96482.0        524.0            15.00     11200.0           Yes   
2   28478.0        638.0             5.40     12100.0            No   
3   25851.0        561.0            17.60      7000.0           Yes   
4   38396.0        527.0             9.80     10700.0            No   
5   69460.5        754.0             3.60     47800.0           Yes   
6  106070.0        665.0            14.60     48000.0            No   
7   35458.0        599.0            21.70     10300.0            No   
8   73520.0        661.0            12.55     17200.0           Yes   
9   57087.0        563.0            11.80     13400.0           Yes   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  
5               No      Yes  
6       

In [ ]:
# 7) Label encoding (Yes/No → 0/1)
# L3: same mapping for target (Approved) and two-category features
# Approved=1, Rejected=0
# --------------------------------
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)
df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [ ]:
# 8) Class balance check
# L3/L5: imbalanced classes can make Accuracy misleading
# --------------------------------
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")


Class balance OK for baseline Accuracy (both classes well represented).


In [ ]:
# 9) Feature engineering (no leakage from label)
# --------------------------------
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

print(df.head(10))

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0             1.10     25800.0              1   
1   96482.0        524.0            15.00     11200.0              1   
2   28478.0        638.0             5.40     12100.0              0   
3   25851.0        561.0            17.60      7000.0              1   
4   38396.0        527.0             9.80     10700.0              0   
5   69460.5        754.0             3.60     47800.0              1   
6  106070.0        665.0            14.60     48000.0              0   
7   35458.0        599.0            21.70     10300.0              0   
8   73520.0        661.0            12.55     17200.0              1   
9   57087.0        563.0            11.80     13400.0              1   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.237111           51814.285714  
1                 0         1      0.116084            6030.125000  


In [ ]:
# 10) MinMaxScaler on numeric features
# L3: scale after outlier capping so mean/std are not skewed by extremes
# Exclude label (Approved) and already-binary columns from scaling
# --------------------------------
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

scaler = MinMaxScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])
print(df.head())


     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.585174     0.121560         0.024490    0.339800              1   
1  0.498215     0.091743         0.591837    0.101287              1   
2  0.018530     0.353211         0.200000    0.115989              0   
3  0.000000     0.176606         0.697959    0.032673              1   
4  0.088490     0.098624         0.379592    0.093118              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.258294               0.846035  
1                 0         1      0.104832               0.082051  
2                 0         1      0.496399               0.055679  
3                 0         1      0.300991               0.004620  
4                 0         1      0.310998               0.040752  


In [ ]:
# 11) Final snapshot + save
# Features (X) = all columns except Approved; label (y) = Approved
# --------------------------------
print("\n=== FINAL HEAD ===")
print(df.head())

print("\n=== FINAL INFO ===")
print(df.info())

print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())


=== FINAL HEAD ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.585174     0.121560         0.024490    0.339800              1   
1  0.498215     0.091743         0.591837    0.101287              1   
2  0.018530     0.353211         0.200000    0.115989              0   
3  0.000000     0.176606         0.697959    0.032673              1   
4  0.088490     0.098624         0.379592    0.093118              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.258294               0.846035  
1                 0         1      0.104832               0.082051  
2                 0         1      0.496399               0.055679  
3                 0         1      0.300991               0.004620  
4                 0         1      0.310998               0.040752  

=== FINAL INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column    

In [ ]:
#savving the data set

df.to_csv("./clean_loan_dataset.csv", index=False)

print("Saved!")

Saved!
